In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_16363/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

In [7]:
lst_s = []
for key, value in enumerate(sqrt_s_lst):
    lst_s.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {lst_s[key]:.2f} GeV^2")

# print(amp_born_lst)


In [8]:
# from scipy.integrate import fixed_quad
# import numpy as np
# from scipy.special import j0

# upper = 10
# s = 10201
# amp = 1347190.7176672097j


# # Inner integral over q from 0 to 1 for a given b
# def inner_integral(b):
#     # Handle both scalar and array inputs
#     if np.isscalar(b):
#         integrand = lambda q: q * j0(b*q)
#         result, _ = fixed_quad(integrand, 0, upper, n=5) 
#         return result * (1/s) * amp
#     else:
#         # For array input, process each element individually
#         results = np.zeros_like(b)
#         for i, b_val in enumerate(b):
#             integrand = lambda q: q * j0(b_val*q)
#             results[i], _ = fixed_quad(integrand, 0, upper, n=5) 
#         return results * (1/s) * amp

# # Outer integrand: b^2 + e^(result of inner integral)
# def outer_integrand(b):
#     inner_result = inner_integral(b)
#     return 1j* s * b * (1- 1j*np.exp(inner_result)) 

# # Outer integral over b from 0 to 1
# def compute_double_integral():
#     result, _ = fixed_quad(outer_integrand, 0, upper, n=5)
#     return result

# # Execute the integration
# result = compute_double_integral()
# print(f"Numerical result: {result:.10f}")


In [9]:
n = 100

inner_upper = 0.1
outer_upper = 10

def inner_integral(b, s, amp):
    # Handle both scalar and array inputs
    if np.isscalar(b):
        integrand = lambda q: q * j0(b*q)
        result, _ = fixed_quad(integrand, 0, inner_upper, n=n) 
        return result * (1/s) * amp
    else:
        # For array input, process each element individually
        results = np.zeros_like(b)
        for i, b_val in enumerate(b):
            integrand = lambda q: q * j0(b_val*q)
            results[i], _ = fixed_quad(integrand, 0, inner_upper, n=n) 
        return results * (1/s) * amp

# Outer integrand: b^2 + e^(result of inner integral)
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j* s * b * (1- 1j*np.exp(inner_result)) 


def compute_double_integral(s, amp):
    result, _ = fixed_quad(lambda b: outer_integrand(b, s, amp), 0, outer_upper, n=n)
    return result

lst_amp_eik = []

for s_val, amp_born_val in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s_val, amp_born_val)
    # print(f"Numerical result: {result1} for s = {s_val} and amp born = {amp_born_val}")
    lst_amp_eik.append(amp_eik_val)

print(lst_amp_eik)


[(47.81697556380686+64.60269217098129j), (414956.58340045105+806404.8260565643j), (1563050.7546788878+3298670.3416857654j), (3382857.8083469803+7540513.834790232j), (5834980.486885393+13566798.424496716j), (8889169.373396663+21401899.217343926j), (12520494.045142535+31064517.058109574j), (16707570.298553385+42569769.51415078j), (21431573.66571257+55930300.011173695j), (26675627.252262887+71156939.66236953j), (32424392.896676093+88259134.72050197j), (38663783.17500549+107245238.9500435j), (45380750.13135722+128122722.52716096j), (52563125.25487948+150898326.52387056j), (60199495.09133497+175578180.37919548j), (68279102.45972422+202167893.30818143j), (76791766.57602984+230672626.82407016j), (85727817.4638465+261097153.23658898j), (95078041.37605658+293445903.5179832j), (104833634.84937441+327723006.9612682j), (114986165.62941796+363932324.40135694j), (125527539.13656087+402077476.3167708j), (136449969.45308578+442161866.8090193j), (147745954.03882238+484188704.22541416j), (159408251.5508

In [10]:
import numpy as np
from scipy.integrate import fixed_quad
from scipy.special import j0

n = 100
lst_inner_upper_limits = [0.1, 0.2, 10]
outer_upper = 10

results = {}  # dictionary to hold results

for inner_upper_val in lst_inner_upper_limits:

    def inner_integral(b, s, amp):
        if np.isscalar(b):
            integrand = lambda q: q * j0(b*q)
            result, _ = fixed_quad(integrand, 0, inner_upper_val, n=n)
            return result * (1/s) * amp
        else:
            results_arr = np.zeros_like(b)
            for i, b_val in enumerate(b):
                integrand = lambda q: q * j0(b_val*q)
                results_arr[i], _ = fixed_quad(integrand, 0, inner_upper_val, n=n)
            return results_arr * (1/s) * amp

    def outer_integrand(b, s, amp):
        inner_result = inner_integral(b, s, amp)
        return 1j * s * b * (1 - 1j * np.exp(inner_result))

    def compute_double_integral(s, amp):
        result, _ = fixed_quad(lambda b: outer_integrand(b, s, amp), 0, outer_upper, n=n)
        return result

    lst_amp_eik = []
    for s_val, amp_born_val in zip(lst_s, amp_born_lst):
        amp_eik_val = compute_double_integral(s_val, amp_born_val)
        lst_amp_eik.append(amp_eik_val)

    # save results in dictionary
    results[inner_upper_val] = lst_amp_eik

# access like:
print(results[0.1])
print(results[0.2])
print(results[10])


[(47.81697556380686+64.60269217098129j), (414956.58340045105+806404.8260565643j), (1563050.7546788878+3298670.3416857654j), (3382857.8083469803+7540513.834790232j), (5834980.486885393+13566798.424496716j), (8889169.373396663+21401899.217343926j), (12520494.045142535+31064517.058109574j), (16707570.298553385+42569769.51415078j), (21431573.66571257+55930300.011173695j), (26675627.252262887+71156939.66236953j), (32424392.896676093+88259134.72050197j), (38663783.17500549+107245238.9500435j), (45380750.13135722+128122722.52716096j), (52563125.25487948+150898326.52387056j), (60199495.09133497+175578180.37919548j), (68279102.45972422+202167893.30818143j), (76791766.57602984+230672626.82407016j), (85727817.4638465+261097153.23658898j), (95078041.37605658+293445903.5179832j), (104833634.84937441+327723006.9612682j), (114986165.62941796+363932324.40135694j), (125527539.13656087+402077476.3167708j), (136449969.45308578+442161866.8090193j), (147745954.03882238+484188704.22541416j), (159408251.5508

In [11]:
def sigma_tot_eik(s, amp):
    return (4 * np.pi) / s * amp.imag * 0.389379323

# dict to store results for each inner_upper_val
results_sigma = {}

for inner_upper_val, lst_amp_eik in results.items():
    lst_sigma_tot_eik = []
    for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
        sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
        lst_sigma_tot_eik.append(sigma_tot_eik_val)

    results_sigma[inner_upper_val] = lst_sigma_tot_eik


In [12]:
# for sigma_tot_eik_val, amp_eik_val in zip(lst_sigma_tot_eik, lst_amp_eik):
#     print(f'amp eik = {amp_eik_val.imag:.1e} = {amp_eik_val.imag}  -  sigma tot eik val = {sigma_tot_eik_val:.2f}')



In [13]:
lst_imag_val_amp_eik = []
for values in lst_amp_eik:
    img_part = values.imag
    lst_imag_val_amp_eik.append(img_part)



In [14]:
def create_iterative_graph(fig, x_data, y_data, title: str, x_axis_name: str, 
                           y_axis_name: str, curve_name: str, line_type, curve_color):

    fig.add_trace(go.Scatter(
        x=x_data,
        y=y_data,
        mode='lines',
        name=curve_name, 
        line=dict(dash=line_type,
                  color=curve_color,
                  width = 3)
    ))

    fig.update_layout(
        title=title,
        xaxis=dict(
            title=x_axis_name,
            type='log',
        ),
        yaxis=dict(
            title=y_axis_name,
            # range=[80, 120]
        ),
        showlegend=True,
        plot_bgcolor='white',
        hovermode='x unified'
    )

    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

In [15]:
imag_parts_01 = np.array(results[0.1]).imag
imag_parts_02 = np.array(results[0.2]).imag
imag_parts_10 = np.array(results[10]).imag

lst_sigma_tot_eik_01 = np.array(results_sigma[0.1])
lst_sigma_tot_eik_02 = np.array(results_sigma[0.2])
lst_sigma_tot_eik_10 = np.array(results_sigma[10])

In [27]:
fig_amp_eik = go.Figure()

create_iterative_graph(fig_amp_eik, sqrt_s_lst, imag_parts_01, 'Amp eikonal vs sqrt', 'sqrt s [GeV]', 'Im(Amp_eikonal)', 
                       'Amp eik - 0.1 limit', 'dot', 'black')

create_iterative_graph(fig_amp_eik, sqrt_s_lst, imag_parts_02, 'Amp eikonal vs sqrt', 'sqrt s [GeV]', 'Im(Amp_eikonal)', 
                       'Amp eik - 0.2 limit', 'solid', 'blue')

create_iterative_graph(fig_amp_eik, sqrt_s_lst, imag_parts_10, 'Amp eikonal vs sqrt', 'sqrt s [GeV]', 'Im(Amp_eikonal)', 
                       'Amp eik - 10 limit', 'dashdot', 'red')

os.makedirs("../../../results/eikonal", exist_ok=True)
file_name = 'amp_eikonal_01_02_10.pdf'
save_path = os.path.join("../../../results/eikonal", file_name)

# fig_amp_eik.show(renderer = 'browser')

fig_amp_eik.write_image(save_path, width=1200, height=600)

In [28]:
fig_sigma_eik = go.Figure()

create_iterative_graph(fig_sigma_eik, sqrt_s_lst, lst_sigma_tot_eik_01, 'Sigma tot eikonal vs sqrt', 'sqrt s [GeV]', 'Sigma tot eikonal [mb]', 
                       'Sigma tot eikonal - 0.1 limit', 'dot', 'black')

create_iterative_graph(fig_sigma_eik, sqrt_s_lst, lst_sigma_tot_eik_02, 'Sigma tot eikonal vs sqrt', 'sqrt s [GeV]', 'Sigma tot eikonal [mb]', 
                       'Sigma tot eikonal - 0.2 limit', 'solid', 'blue')

create_iterative_graph(fig_sigma_eik, sqrt_s_lst, lst_sigma_tot_eik_10, 'Sigma tot eikonal vs sqrt', 'sqrt s [GeV]', 'Sigma tot eikonal [mb]', 
                       'Sigma tot eikonal - 10 limit', 'dashdot', 'red')

os.makedirs("../../../results/eikonal", exist_ok=True)
file_name = 'sigma_tot_eikonal_01_02_10.pdf'
save_path = os.path.join("../../../results/eikonal", file_name)

# fig_sigma_eik.show(renderer = 'browser')

fig_sigma_eik.write_image(save_path, width=1200, height=600)

In [23]:
print(sqrt_s_lst)

[1, 101, 201, 301, 401, 501, 601, 701, 801, 901, 1001, 1101, 1201, 1301, 1401, 1501, 1601, 1701, 1801, 1901, 2001, 2101, 2201, 2301, 2401, 2501, 2601, 2701, 2801, 2901, 3001, 3101, 3201, 3301, 3401, 3501, 3601, 3701, 3801, 3901, 4001, 4101, 4201, 4301, 4401, 4501, 4601, 4701, 4801, 4901, 5001, 5101, 5201, 5301, 5401, 5501, 5601, 5701, 5801, 5901, 6001, 6101, 6201, 6301, 6401, 6501, 6601, 6701, 6801, 6901, 7001, 7101, 7201, 7301, 7401, 7501, 7601, 7701, 7801, 7901, 8001, 8101, 8201, 8301, 8401, 8501, 8601, 8701, 8801, 8901, 9001, 9101, 9201, 9301, 9401, 9501, 9601, 9701, 9801, 9901, 10001, 10101, 10201, 10301, 10401, 10501, 10601, 10701, 10801, 10901, 11001, 11101, 11201, 11301, 11401, 11501, 11601, 11701, 11801, 11901, 12001, 12101, 12201, 12301, 12401, 12501, 12601, 12701, 12801, 12901]
